# Taxxa — Embed `nodes.json` on Colab GPU (Drive-mounted)

Reads `nodes.json` from `MyDrive/prompt/` and writes `vectors.npy` + `id_map.json` back to the same folder. Matches `embedder.py` exactly.

**Before running**: Runtime → Change runtime type → T4 GPU (or better).

**Expected paths**:
- Input:  `/content/drive/MyDrive/prompt/nodes.json`
- Output: `/content/drive/MyDrive/prompt/vectors.npy`
- Output: `/content/drive/MyDrive/prompt/id_map.json`

In [ ]:
# 1. Confirm GPU is attached
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU detected. Runtime → Change runtime type → T4 GPU.')

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Verify paths
import os
DRIVE_DIR = '/content/drive/MyDrive/prompt'
NODES_PATH = os.path.join(DRIVE_DIR, 'nodes.json')
VECTORS_PATH = os.path.join(DRIVE_DIR, 'vectors.npy')
ID_MAP_PATH = os.path.join(DRIVE_DIR, 'id_map.json')

assert os.path.isdir(DRIVE_DIR), f'Folder not found: {DRIVE_DIR}'
assert os.path.isfile(NODES_PATH), f'File not found: {NODES_PATH}'
print(f'nodes.json: {os.path.getsize(NODES_PATH) / 1024 / 1024:.1f} MB')
print(f'Writing outputs to: {DRIVE_DIR}')

In [ ]:
# 4. Install sentence-transformers (Colab usually has torch already)
!pip install -q sentence-transformers

In [ ]:
# 5. Load nodes from Drive
import json
with open(NODES_PATH, encoding='utf-8') as f:
    nodes = json.load(f)
print(f'Loaded {len(nodes):,} nodes')
print(f'Sample: {nodes[0]["id"][:80]}')

In [ ]:
# 6. Embed — matches embedder.py exactly
from sentence_transformers import SentenceTransformer
import numpy as np
import time

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

print(f'Loading model: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME, device='cuda')
# Match local embedder.py: 256 covers ~95% of chunks without truncation
model.max_seq_length = 256

# Same input format as local: title + chunk text
texts = [f"{n['title']}\n\n{n['text']}" for n in nodes]
print(f'Embedding {len(texts):,} chunks on GPU (batch_size=256)...')

t0 = time.time()
vectors = model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
elapsed = time.time() - t0
print(f'Done in {elapsed/60:.1f} min — shape={vectors.shape}, dim={vectors.shape[1]}')

In [ ]:
# 7. Save vectors + id_map directly to Drive
np.save(VECTORS_PATH, vectors)
id_map = {i: n['id'] for i, n in enumerate(nodes)}
with open(ID_MAP_PATH, 'w', encoding='utf-8') as f:
    json.dump(id_map, f, ensure_ascii=False)

print(f'Wrote {VECTORS_PATH} ({os.path.getsize(VECTORS_PATH) / 1024 / 1024:.1f} MB)')
print(f'Wrote {ID_MAP_PATH} ({os.path.getsize(ID_MAP_PATH) / 1024 / 1024:.1f} MB)')
print('\nDone. Sync Drive locally and copy both files into taxxa/data/')

## After running

Both files are now in your Google Drive at `MyDrive/prompt/`. Pull them to your local machine:

```bash
# If you have Drive synced via Drive for Desktop:
cp ~/Library/CloudStorage/GoogleDrive-*/My\ Drive/prompt/vectors.npy ~/projects/taxxa/data/
cp ~/Library/CloudStorage/GoogleDrive-*/My\ Drive/prompt/id_map.json ~/projects/taxxa/data/
```

Or download via the Drive web UI and move them into `taxxa/data/`, then:

```bash
cd ~/projects/taxxa && source .venv/bin/activate && python3 test_retrieval.py
```